In [ ]:
import arcpy
from datetime import datetime, timedelta
import os

# --- 1. CONFIGURATION ---
gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_stacked_removed")

In [ ]:
# ===================================================================
# create buffers
# ===================================================================
import arcpy
import os

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_stacked_removed")

# Enable overwrite so you can test safely
arcpy.env.overwriteOutput = True

# Your exact requested testing intervals
buffer_distances = {
    "0_5km": "500 Meters",
    "1km":   "1000 Meters",
    "1_5km": "1500 Meters",
    "2km":   "2000 Meters",
    "2_5km": "2500 Meters",
    "3km":   "3000 Meters",
    "3_5km": "3500 Meters",
    "4km":   "4000 Meters"
}

# Run the physical buffering exactly like the FOD notebook
for label, dist in buffer_distances.items():
    out_fc = os.path.join(gdb, f"Permit_buffer_{label}")
    print(f"Generating physical permit buffer layer for {label} ({dist})...")
    
    arcpy.analysis.Buffer(
        in_features=permits,
        out_feature_class=out_fc,
        buffer_distance_or_field=dist,
        dissolve_option="NONE" # Preserves individual permit attributes for the join
    )

print("\nAll permit buffer layers generated successfully.")

In [ ]:
#create permit labels in new layer

import arcpy
import os

gdb           = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
events_source = os.path.join(gdb, "SEFM_events_94_24")
events_target = os.path.join(gdb, "SEFM_events_94_24_permit_labeled")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

# --- STEP 1: CLONE THE MASTER SEFM LAYER ---
print(f"Creating dedicated working copy: {os.path.basename(events_target)}...")
arcpy.management.CopyFeatures(events_source, events_target)

# --- STEP 2: INITIALIZE CLASSIFICATION FIELDS ---
buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]
fields_to_add = []
existing_fields = [f.name for f in arcpy.ListFields(events_target)]

for b in buffer_sizes:
    field_name = f"permit_{b}"
    if field_name not in existing_fields:
        # Format: [Field Name, Field Type, Field Alias, Field Length]
        fields_to_add.append([field_name, "TEXT", field_name, 20])

if fields_to_add:
    print(f"Adding {len(fields_to_add)} clean permit tracking fields...")
    arcpy.management.AddFields(events_target, fields_to_add)

print("-" * 60)
print("SUCCESS: Dedicated permit classification layer is ready.")
print("-" * 60)

In [ ]:
# -------------------------------------------------------------------
# 2. DATE PARSERS (UPDATED WITH EXPLICIT CLASS IMPORT)
# -------------------------------------------------------------------
from datetime import datetime  # Ensures 'datetime' is recognized as a type

def parse_sefm_date(value):
    """Convert SEFM YYYYMMDD integer/string into datetime."""
    if value is None:
        return None
    return datetime.strptime(str(value), "%Y%m%d")


def parse_permit_date(value):
    """Ensure permit date is a clean datetime object."""
    if value is None:
        return None
    
    # Now that the class is imported correctly, isinstance will work perfectly
    if isinstance(value, datetime):
        return value
        
    # Fallback safety check for string variants
    try:
        return datetime.strptime(str(value).strip().split(" ")[0], "%Y-%m-%d")
    except ValueError:
        return None

In [ ]:
# -------------------------------------------------------------------
# BUILD PERMIT DATE LOOKUP WITH PADDED WINDOWS (+/- 30 DAYS)
# -------------------------------------------------------------------
from datetime import timedelta

print("Building memory-resident Permit date lookup dictionary...")
permit_dates = {}

# We are using your new permanent 'permit_id' alongside the true DATE fields
fields = ["permit_id", "start_date", "end_date"]

with arcpy.da.SearchCursor(permits, fields) as cur:
    for pid, s_date, e_date in cur:
        # Pass the values through our updated native date object validator
        dt_start = parse_permit_date(s_date)
        dt_end = parse_permit_date(e_date)
        
        if dt_start is None or dt_end is None:
            continue

        # Create the expanded temporal acceptance window around the permit's lifespan
        permit_dates[pid] = {
            "min": dt_start - timedelta(days=30),
            "max": dt_end + timedelta(days=30)
        }

print(f"Lookup dictionary complete. Indexed {len(permit_dates):,} permit records.")

In [ ]:
# -------------------------------------------------------------------
# 6. SPATIAL + TEMPORAL MATCHING FOR EACH PERMIT BUFFER SIZE
# -------------------------------------------------------------------
import os

# Define the dictionary explicitly so the kernel has it in memory
buffer_distances = {
    "0_5km": "500 Meters",
    "1km":   "1000 Meters",
    "1_5km": "1500 Meters",
    "2km":   "2000 Meters",
    "2_5km": "2500 Meters",
    "3km":   "3000 Meters",
    "3_5km": "3500 Meters",
    "4km":   "4000 Meters"
}

# Ensure we use your dedicated permit layer as the operational target
events_permit_layer = os.path.join(gdb, "SEFM_events_94_24_permit_labeled")

for label in buffer_distances.keys():

    print(f"Processing permit buffer {label}...")

    # Dynamic pathing matching our setup variables
    buffer_fc = os.path.join(gdb, f"Permit_buffer_{label}")
    sj        = os.path.join(gdb, f"sj_permit_{label}")

    # Spatial join: Target = SEFM permit events, Join = physical permit buffer circles
    arcpy.analysis.SpatialJoin(
        target_features=events_permit_layer,
        join_features=buffer_fc,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # event_id → list of permit_ids intersecting it
    event_to_permit = {}

    with arcpy.da.SearchCursor(sj, ["event_id", "permit_id"]) as cur:
        for eid, pid in cur:
            if pid == -1 or pid is None:
                continue
            event_to_permit.setdefault(eid, []).append(pid)

    # Match the dynamic tracking field name we appended earlier
    field = f"permit_{label}"

    with arcpy.da.UpdateCursor(
        events_permit_layer,
        ["event_id", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for row in cur:
            eid, tmin_raw, tmax_raw, current_val = row

            # If a smaller buffer run already classified this as a prescribed fire, 
            # skip it so we don't accidentally overwrite it back to None/NULL!
            if current_val == "prescribed_fire":
                continue

            # Parse SEFM dates
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            if tmin is None or tmax is None:
                # Force to NULL if dates are missing/corrupted
                row[3] = None
                cur.updateRow(row)
                continue

            classification = None

            # Only evaluate if a spatial match exists for this event
            if eid in event_to_permit:

                for pid in event_to_permit[eid]:

                    p_window = permit_dates.get(pid)
                    if p_window is None:
                        continue

                    permit_min = p_window["min"]
                    permit_max = p_window["max"]

                    # Temporal overlap test (Padded Permit start <= Satellite end AND Padded Permit end >= Satellite start)
                    if permit_max >= tmin and permit_min <= tmax:
                        classification = "prescribed_fire"
                        break

            # Explicitly route matches to 'prescribed_fire' and non-matches to database NULL
            if classification == "prescribed_fire":
                row[3] = "prescribed_fire"
            else:
                row[3] = None
                
            cur.updateRow(row)

    # Clean up the intermediate spatial join table
    if arcpy.Exists(sj):
        arcpy.management.Delete(sj)

# Clean cache to prevent map lockouts
arcpy.management.ClearWorkspaceCache(gdb)

# Native ArcGIS Pro window mapping interface refresh
try:
    arcpy.Project("CURRENT").importWorkspace(gdb)
except Exception:
    pass  # Standalone script fallback safety

print("Spatio-temporal permit classification complete.")

In [ ]:
# ===================================================================
# Calculate the percentage of total burn permits that 
#          matched an SEFM satellite event at each buffer size.
# ===================================================================
import arcpy
import os

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_stacked_removed")
events  = os.path.join(gdb, "SEFM_events_94_24_permit_labeled")

# Enable overwrite capability so it clears out temp tables dynamically
arcpy.env.overwriteOutput = True

# Get the baseline total of unique burn permits
total_permits = int(arcpy.management.GetCount(permits)[0])
print(f"Total baseline burn permits: {total_permits:,}\n")

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]
results = {}

for label in buffer_sizes:

    print(f"Processing permit buffer {label}...")

    buffer_fc = os.path.join(gdb, f"Permit_buffer_{label}")
    sj        = os.path.join(gdb, f"permit_match_summary_{label}")

    # Spatial join: Permit buffers (Target) → SEFM labeled events (Join)
    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=events,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # SAFETY CHECK: Resolve exact field name variations if Pro appends '_1'
    actual_fields = [f.name for f in arcpy.ListFields(sj)]
    expected_field = f"permit_{label}"
    
    if expected_field not in actual_fields and f"{expected_field}_1" in actual_fields:
        class_field = f"{expected_field}_1"
    else:
        class_field = expected_field

    matched_permit_ids = set()

    # Read classification and pull unique matched permit IDs into memory
    with arcpy.da.SearchCursor(sj, ["permit_id", class_field]) as cur:
        for pid, classification in cur:
            if classification == "prescribed_fire":
                matched_permit_ids.add(pid)

    # Calculate the percentage of unique permits that found a match
    if total_permits > 0:
        pct = (len(matched_permit_ids) / total_permits) * 100
    else:
        pct = 0.0
        
    results[label] = pct
    print(f"-> {label}: {pct:.2f}% of permits matched")

    # Clean up intermediate join tables immediately to prevent GDB bloating
    if arcpy.Exists(sj):
        arcpy.management.Delete(sj)

# --- CLEAN SUMMARY OUTPUT ---
print("\n" + "="*45)
print("FINAL SUMMARY: BURN PERMIT MATCH PERCENTAGES")
print("="*45)
for label, pct in results.items():
    print(f" Buffer {label:<6}: {pct:.2f}% of permits matched an event")
print("="*45)

In [ ]:
# transfer 1.5 km buffer permit classifications to master layer
import arcpy
import os

# --- PATHS ---
gdb       = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
source_fc   = os.path.join(gdb, "SEFM_events_94_24_permit_labeled")
target_fc   = os.path.join(gdb, "SEFM_events_94_24")

# --- 1. AUTOMATICALLY DETECT FIELD NAME ---
source_fields = [f.name for f in arcpy.ListFields(source_fc)]

if "permit_1_5km" in source_fields:
    source_field = "permit_1_5km"
elif "permit_1_5km_1" in source_fields:
    source_field = "permit_1_5km_1"
else:
    raise ValueError("Could not find permit_1_5km or permit_1_5km_1 in the source layer! Double check your fields.")

target_field = "permit_match"
print(f"Found source field: {source_field}. Reading matches into memory...")

# --- 2. READ MATCHES INTO MEMORY ---
match_lookup = {}
with arcpy.da.SearchCursor(source_fc, ["event_id", source_field]) as cur:
    for eid, val in cur:
        if eid is not None and val is not None:
            match_lookup[eid] = val

print(f"Loaded {len(match_lookup):,} matched event records.")

# --- 3. CREATE TARGET FIELD IF MISSING ---
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if target_field not in existing_fields:
    print(f"Adding new field '{target_field}' to master layer...")
    arcpy.management.AddField(target_fc, target_field, "TEXT", field_length=20)

# --- 4. TRANSFER THE DATA ---
print(f"Writing values over to {target_field}...")
update_count = 0

with arcpy.da.UpdateCursor(target_fc, ["event_id", target_field]) as cur:
    for row in cur:
        eid = row[0]
        if eid in match_lookup:
            row[1] = match_lookup[eid]
            cur.updateRow(row)
            update_count += 1

# Flush cache so ArcGIS Pro visualizes the changes immediately
arcpy.management.ClearWorkspaceCache(gdb)
print(f"Success. Updated {update_count:,} rows. Check SEFM_events_94_24.")

In [ ]:
# -------------------------------------------------------------------
# FIX AND STRICTLY MATCH 1.5KM PERMIT DATES 
#-------------------------------------------------------------------
import arcpy
import os
from datetime import timedelta, datetime

# --- PATHS ---
gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits      = os.path.join(gdb, "burn_permits_stacked_removed")
exist_buffer = os.path.join(gdb, "Permit_buffer_1_5km")
target_fc    = os.path.join(gdb, "SEFM_events_94_24")

# Enable overwrite capability
arcpy.env.overwriteOutput = True

print("Step 1: Cleaning display layout and removing old fields...")
for old_field in ["permit_start_date", "permit_end_date", "Permit Start Date", "Permit End Date"]:
    if old_field in [f.name for f in arcpy.ListFields(target_fc)]:
        arcpy.management.DeleteField(target_fc, old_field)

print("Step 2: Adding clean snake_case fields without active display aliases...")
# Forcing Field Name and Alias to be identical ensures ArcGIS Pro shows permit_start_date
arcpy.management.AddFields(target_fc, [
    ["permit_start_date", "DATE", "permit_start_date"],
    ["permit_end_date", "DATE", "permit_end_date"]
])

print("Step 3: Indexing raw permit dates and applying +/- 30 day padding windows...")
permit_date_lookup = {}
with arcpy.da.SearchCursor(permits, ["permit_id", "start_date", "end_date"]) as cur:
    for pid, s_date, e_date in cur:
        dt_start = parse_permit_date(s_date)
        dt_end = parse_permit_date(e_date)
        
        if dt_start is not None and dt_end is not None:
            # Force timeline properties into pure date-objects (stripping midnight time markers)
            permit_date_lookup[pid] = {
                "clean_start": dt_start.date() if isinstance(dt_start, datetime) else dt_start,
                "clean_end": dt_end.date() if isinstance(dt_end, datetime) else dt_end,
                "padded_min": dt_start - timedelta(days=30),
                "padded_max": dt_end + timedelta(days=30)
            }

print("Step 4: Running localized spatial index matrix...")
temp_sj = os.path.join(gdb, "temp_strict_date_sj")
arcpy.analysis.SpatialJoin(
    target_features=target_fc,
    join_features=exist_buffer,
    out_feature_class=temp_sj,
    join_operation="JOIN_ONE_TO_MANY",
    match_option="INTERSECT"
)

print("Step 5: Grouping spatial overlaps by event ID...")
spatial_map = {}
with arcpy.da.SearchCursor(temp_sj, ["event_id", "permit_id"]) as cur:
    for eid, pid in cur:
        if pid != -1 and pid is not None and eid is not None:
            spatial_map.setdefault(eid, []).append(pid)

print("Step 6: Executing strict spatio-temporal validation and writing dates...")
update_count = 0
mismatch_prevented = 0

fields_to_update = ["event_id", "permit_match", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", "permit_start_date", "permit_end_date"]
with arcpy.da.UpdateCursor(target_fc, fields_to_update) as cur:
    for row in cur:
        eid, permit_match, tmin_raw, tmax_raw = row[0], row[1], row[2], row[3]
        
        if permit_match == "prescribed_fire" and eid in spatial_map:
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)
            
            if tmin is None or tmax is None:
                continue
            
            found_valid_date_match = False
            
            # Look through all permits that share this coordinate spot
            for pid in spatial_map[eid]:
                if pid in permit_date_lookup:
                    pdata = permit_date_lookup[pid]
                    
                    # STRICT TIMELINE CHECK: Padded Permit start <= Satellite end AND Padded Permit end >= Satellite start
                    if pdata["padded_max"] >= tmin and pdata["padded_min"] <= tmax:
                        row[4] = pdata["clean_start"]
                        row[5] = pdata["clean_end"]
                        cur.updateRow(row)
                        update_count += 1
                        found_valid_date_match = True
                        break  # Option A: Take the first permit that passes both spatial AND temporal rules
            
            if not found_valid_date_match:
                mismatch_prevented += 1

# --- STEP 7: CLEANUP ---
if arcpy.Exists(temp_sj):
    arcpy.management.Delete(temp_sj)

arcpy.management.ClearWorkspaceCache(gdb)
print("\n" + "="*60)
print(f"SUCCESS: Processed clean dates fields.")
print(f"-> Successfully written matched dates: {update_count:,} rows")
print(f"-> Out-of-year mismatched permits rejected: {mismatch_prevented:,} rows")
print("="*60)